In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import xarray as xr
from glob import glob
import cartopy.crs as ccrs
from FUNCTIONS import get_dz, tracer_load, lat_lon_cell_area
from matplotlib import ticker

## Load the data
Insert the path to your output

In [ ]:
# Enter path to output directory
datadir = '/path/to/output/data/'
ds = xr.open_mfdataset(datadir+'Raikoke-June-2019-forecast_mode-remap_DOM01_ML*',autoclose=True)
dz  = get_dz(ds.isel(time=0))

In [ ]:
# for conversion mol/mol --> kg/kg
MM_SO2 = 64/28.96 # molar weight of SO2 / dry air
MM_H2SO4 = 98/28.96 

## Print Dataset

In [ ]:
ds

## Sellect tracers
Insert the names of the tracers 

In [ ]:
SO2 = ds['???']
SO2 = SO2.where(SO2>1e-9)

H2SO4 = ds['???']
H2SO4 = H2SO4.where(H2SO4>1e-20)

insol_ash = ds['???'] + ds['???'] + ds['???']
insol_ash = insol_ash.where(insol_ash>10)

mixed_ash = ds['???'] + ds['???']
mixed_ash = mixed_ash.where(mixed_ash>1)

sulfate = ds['???'] + ds['???']
sulfate = sulfate.where(sulfate>1e-2)

sulf_aero = ds['???'] + ds['???']
sulf_aero = sulf_aero.where(mixed_ash>1)

In [ ]:
# Tracer load of SO2 
SO2_load = tracer_load(SO2*MM_SO2,ds['rho'],dz)

# Tracer load of H2SO4  
H2SO4_load = tracer_load(H2SO4*MM_H2SO4,ds['rho'],dz)

# Tracer load of insoluble ash tracers
insol_load = tracer_load(insol_ash,ds['rho'],dz)*1e-9

# Tracer load of mixed ash tracers
mixed_load = tracer_load(mixed_ash,ds['rho'],dz)*1e-9

# Tracer load of all insoluble ash tracers
sulfate_load = tracer_load(sulfate,ds['rho'],dz)*1e-9

# Tracer load of mixed ash tracers
sulf_aero_load = tracer_load(sulf_aero,ds['rho'],dz)*1e-9

## Plot tracer loading maps in kg/m2 after 24h

In [ ]:
# Plot SO2 column 
log_lev1=np.array([1e-9,1e-8,1e-7,1e-6,1e-5,1e-4,1e-3,1e-2,1e-1])
fig1 = plt.figure(figsize=(12,8))

ax1=fig1.add_subplot(3,2,1,projection=ccrs.PlateCarree(central_longitude=180))
plot1=ax1.contourf(ds.lon, ds.lat, SO2_load.isel(time=24), locator=ticker.LogLocator(), levels=log_lev1, extend='max', cmap='Blues', transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('SO2')

ax1=fig1.add_subplot(3,2,2,projection=ccrs.PlateCarree(central_longitude=180))
plot2=ax1.contourf(ds.lon, ds.lat, H2SO4_load.isel(time=24), locator=ticker.LogLocator(), levels=log_lev1, extend='max', cmap='Blues', transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('H2SO4')

ax1=fig1.add_subplot(3,2,3,projection=ccrs.PlateCarree(central_longitude=180))
plot3=ax1.contourf(ds.lon, ds.lat, insol_load.isel(time=24), locator=ticker.LogLocator(), levels=log_lev1, extend='max', cmap='Blues', transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('Insoluble ash')

ax1=fig1.add_subplot(3,2,4,projection=ccrs.PlateCarree(central_longitude=180))
plot4=ax1.contourf(ds.lon, ds.lat, mixed_load.isel(time=24), locator=ticker.LogLocator(), levels=log_lev1, extend='max', cmap='Blues', transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('Mixed Ash')

ax1=fig1.add_subplot(3,2,5,projection=ccrs.PlateCarree(central_longitude=180))
plot5=ax1.contourf(ds.lon, ds.lat, sulfate_load.isel(time=24), locator=ticker.LogLocator(), levels=log_lev1, extend='max', cmap='Blues', transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('Sulfate')

ax1=fig1.add_subplot(3,2,6,projection=ccrs.PlateCarree(central_longitude=180))
plot6=ax1.contourf(ds.lon, ds.lat, sulf_aero_load.isel(time=24), locator=ticker.LogLocator(), levels=log_lev1, extend='max', cmap='Blues', transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('Sulfate on ash aerosols')

ax2=fig1.add_axes([0.3,0.05,0.4,0.03])
cbar=plt.colorbar(plot1,cax=ax2,orientation='horizontal',format="%0.0e")
cbar.set_label('Column loading in kg/m2')

## Effect of the OH chemistry on SO2 and sulfate

In [ ]:
cell_area = lat_lon_cell_area(ds.lat,ds.lon)
time = np.arange(0,49)

In [ ]:
SO2_mass       = (SO2_load * cell_area).compute()
H2SO4_mass     = (H2SO4_load * cell_area).compute()
so4_mass       = (sulfate_load * cell_area).compute()
so4_aero_mass  = (sulf_aero_load * cell_area).compute()

In [ ]:
fig1 = plt.figure(figsize=(15,5))

ax1=fig1.add_subplot(1,3,1)
plt.plot(time,SO2_mass.sum(dim=['lat', 'lon']))
plt.title('SO2 mass')

ax1=fig1.add_subplot(1,3,2)
plt.plot(time,H2SO4_mass.sum(dim=['lat', 'lon']))
plt.title('H2SO4 mass')

ax1=fig1.add_subplot(1,3,3)
plt.plot(time,(so4_mass+so4_aero_mass).sum(dim=['lat', 'lon']))
plt.title('Sulfate aerosol mass')


In [ ]:
OH_conc = ds['OH_Nconc'].isel(height=89)

fig1 = plt.figure(figsize=(20,6))

ax1=fig1.add_subplot(2,2,1,projection=ccrs.PlateCarree(central_longitude=180))
plot1=ax1.contourf(ds.lon, ds.lat, OH_conc.isel(time=12),levels=np.arange(0,1e7,1e6),cmap='plasma_r',extend='max',transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('OH surface concentration in #/cm3 after 12 h')
plt.colorbar(plot1)

ax1=fig1.add_subplot(2,2,2,projection=ccrs.PlateCarree(central_longitude=180))
plot1=ax1.contourf(ds.lon, ds.lat, OH_conc.isel(time=18),levels=np.arange(0,1e7,1e6),cmap='plasma_r',extend='max',transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('OH surface concentration in #/cm3 after 18 h')
plt.colorbar(plot1)

ax1=fig1.add_subplot(2,2,3,projection=ccrs.PlateCarree(central_longitude=180))
plot1=ax1.contourf(ds.lon, ds.lat, OH_conc.isel(time=24),levels=np.arange(0,1e7,1e6),cmap='plasma_r',extend='max',transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('OH surface concentration in #/cm3 after 24 h')
plt.colorbar(plot1)

ax1=fig1.add_subplot(2,2,4,projection=ccrs.PlateCarree(central_longitude=180))
plot1=ax1.contourf(ds.lon, ds.lat, OH_conc.isel(time=30),levels=np.arange(0,1e7,1e6),cmap='plasma_r',extend='max',transform = ccrs.PlateCarree())
ax1.coastlines()
plt.title('OH surface concentration in #/cm3 after 30 h')
plt.colorbar(plot1)